## Using edgeR for DEG analysis between clusters -- comparing region_leiden clusters within capillary cells and using pseurod-bulk approach

In [ ]:
### identify cluster marker genes
suppressPackageStartupMessages({
    library(Seurat)
    library(stringr)
    library(dplyr)
    library(patchwork)
    library(ggplot2)
    library(future)
    library(tidydr)
    library(harmony)
    library(reticulate)
    library(viridis)
    library(RColorBrewer)
    library(ComplexHeatmap)
    library(colorRamp2)
    library(edgeR)
})

# set large memory  
options(future.globals.maxSize = 120*1024^3)

## plotting parameters
umap_theme <- theme_dr()+theme(panel.grid.major = element_blank(), 
                                            panel.grid.minor = element_blank(),
                                            panel.background = element_blank(), 
                                            axis.line = element_line(colour = "black"))

## set working dir
dir = "/home/kibr/Work/Vascular_atlas/"
setwd(dir)

In [ ]:
# obj_oi = readRDS("./Results/Revision_2/Endothelial.rds")
## Reloading 
# Convert h5ad to h5seurat
h5ad_file="./Results/Revision_2/Fibroblast_temp_processed.h5ad"
obj_oi = schard::h5ad2seurat(h5ad_file)
print(obj_oi)

In [ ]:
### optional: check how many brain regions are there
length(unique(obj_oi$brain_region))
table(obj_oi$region_abb)

In [ ]:
## check cluster sizes and combine small clusters if needed
table(obj_oi$Cell_type,useNA = "always")

# table(obj_oi$region_cluster,useNA = "always")

In [ ]:
celltype_colors <- c(
  "Fib_1" = "#00695C",
  "Fib_2" = "#00796B",
  "Fib_3" = "#009688",
  "Fib_4" = "#26A69A",
  "Fib_5" = "#4DB6AC",
  "Fib_6" = "#80CBC4"
)

region_color =c(
    "Cortex" =  "#009E73",
    "Brainstem" = "#D55E00",
    "Cerebellum" = "#046299",
    "Limbic" = "#03B8D8",
    "Watershed" = "#E69F00",
    "White Matter Tracts" = "#CC79A7",
    "Olfactory" = "#999999",
    "Barrier" = "#9C0AE0",
    # "Subcortical" = "#915330",
    "Major Vessel" = "#FF0000"
)

In [ ]:
umap_theme <- theme_dr()+theme(panel.grid.major = element_blank(), 
                                            panel.grid.minor = element_blank(),
                                            panel.background = element_blank(), 
                                            axis.line = element_line(colour = "black"))

options(repr.plot.width = 6, repr.plot.height = 5, repr.plot.res = 300)
p1 <- DimPlot(obj_oi,group.by="Cell_type",reduction = "umapharmony_",cols = celltype_colors,raster = F,pt.size = 1.5,label = T)+umap_theme+ggtitle("Cell_Type")
# p2 <- DimPlot(obj_oi,group.by="harmony_clusters",repel = TRUE,label = T,reduction = "umapharmony_")+NoLegend()+umap_theme+ggtitle("cluster")
p3 <- DimPlot(obj_oi, label = F,  reduction = "umapharmony_",group.by = "region_layer",cols = region_color,raster = F,pt.size = 1.5)+umap_theme+ggtitle("brain_region")

In [ ]:
options(repr.plot.width = 6, repr.plot.height = 5, repr.plot.res = 300)
p1
options(repr.plot.width = 7, repr.plot.height = 5, repr.plot.res = 300)
p3

In [ ]:
ggsave(filename = "./Results/Revision_2/Figures/Figure_4X_Fibroblast_cell_type_UMAP.pdf",
    patchwork::wrap_plots(p1,p3,ncol = 1),
      scale = 1, width = 12, height = 14)

In [ ]:
## exclude samples with less than 3 cells
meta = obj_oi@meta.data
mat = table(meta$sampleID)
sample_excluded = names(mat[mat<3])
length(sample_excluded)

obj_oi = subset(obj_oi,subset = !sampleID %in% sample_excluded)
obj_oi

In [ ]:
## further filter genes with less expression
min_features_pct =0.05
min_features_ncell = 20

##--- Get the expression summary ---##
# obj_oi <- JoinLayers(obj_oi)
# expr = LayerData(obj_oi, assay = "decontX", layer = "counts")
expr = LayerData(obj_oi, assay = "RNA", layer = "counts")

## calculate percentage of cells expressing each gene
genes.percent.expression <- rowMeans(expr>0)
# genes.percent.expression = rowSums(expr>0)

## select genes based on expression threshold
genes = names(genes.percent.expression[genes.percent.expression>min_features_pct])
# genes = names(genes.percent.expression[genes.percent.expression>min_features_ncell])

print(paste(length(genes),"genes will be tested."))

## Using wilcoxon rank sum test to identify cluster marker genes

In [ ]:
# Normalize the data
obj_oi <- NormalizeData(obj_oi)
Idents(obj_oi) = obj_oi$Cell_type
obj_oi

In [ ]:
all.markers <- FindAllMarkers(object = obj_oi,verbose = T,min.pct = 0.25,logfc.threshold = 0.25
#,test.use = "MAST" #might use MAST model
) 
all.markers %>% 
    group_by(cluster) %>% 
    filter(p_val_adj < 0.05, avg_log2FC > 0.20) %>%
    top_n(n = 5, wt = avg_log2FC) -> top

all.markers %>% 
    group_by(cluster) %>% 
    filter(p_val_adj < 0.05, avg_log2FC > 0) -> sig

In [ ]:
table(all.markers$cluster)

In [ ]:
write.csv(all.markers,file = "./Results/Revision_2/DEG/Table_S8_Fibroblast_cell_type_markers.csv")

In [ ]:
## scale the gene expression of seurat object data for heatmap plot
obj_oi <- ScaleData(object = obj_oi, features = genes)

In [ ]:
p1 <-  DotPlot(obj_oi,cols = viridis(3, option = "magma"),
             features = unique(top$gene)) + RotatedAxis(180) +
                                    theme(axis.title = element_blank(),legend.position = "top")
p1

## Get psudo-bulk expression

In [ ]:
## getting the pseudo-bulk object
pseudo_obj <- AggregateExpression(obj_oi, assays = "RNA", return.seurat = T, group.by = c("sampleID","Cell_type"))
pseudo_obj

In [ ]:
mat = pseudo_obj[['RNA']]$counts
head(mat)

In [ ]:
meta = pseudo_obj@meta.data

group = meta$Cell_type
head(meta)

In [ ]:
## Build edgeR format
y <- DGEList(counts=mat,group=group)

## add donor info
df = str_split_fixed(rownames(y$samples),"-",4)[,c(1,2)]
y$samples$donor = paste(df[,1],df[,2],sep = "_")

## filter sample by number of reads
summary(y$samples$lib.size)
keep.samples <- y$samples$lib.size > 5e3 #number of reads>5000
table(keep.samples)
y <- y[, keep.samples]

## preprocessing for model fitting
group <- droplevels(factor(y$samples$group))
donor <- droplevels(factor(y$samples$donor))

design <- model.matrix(~0 + group + donor)
colnames(design) <- make.names(colnames(design))
head(design)

## filter genes based on design
keep <- filterByExpr(y, design=design)
table(keep)
y <- y[keep, , keep.lib.sizes=FALSE]

In [ ]:
## normalize library sizes
y <- normLibSizes(y)
summary(y$samples$norm.factors)

options(repr.plot.width=12,repr.plot.height=12)
brain_region <- as.factor(y$samples$group)
plotMDS(y, pch=16, col=c(1:36)[y$samples$group], main="brain region",labels = y$samples$donor)

In [ ]:
## estimate dispersion
y <- estimateDisp(y, design, robust=TRUE)
##--------- fit the model -----------##
fit <- glmQLFit(y, design, robust=TRUE)

In [ ]:

K <- nlevels(group)
grp_cols <- grep("^group", colnames(design), value=TRUE)

# sanity: length(grp_cols) should be K
stopifnot(length(grp_cols) == K)

C <- matrix(-1/(K-1), nrow=K, ncol=K,
            dimnames=list(grp_cols, levels(group)))
diag(C) <- 1

# expand to full design rows (add zeros for donor columns)
Cfull <- matrix(0, nrow=ncol(design), ncol=K,
                dimnames=list(colnames(design), colnames(C)))
Cfull[grp_cols, ] <- C

qlf <- vector("list", K)
for(i in seq_len(K)) {
  qlf[[i]] <- glmQLFTest(fit, contrast=Cfull[, i])
  qlf[[i]]$comparison <- paste0(levels(group)[i], "_vs_others")
}


In [ ]:
res = data.frame()
for (i in 1:length(qlf)){
    df = qlf[[i]]$table
    df$genes = rownames(df)
    df$FDR = p.adjust(df$PValue, method = "bonferroni")
    df %>%
        dplyr::arrange(FDR) %>%
        # dplyr::filter(FDR  < 0.05) %>%
        # dplyr::filter(logFC > 0) %>% 
        ungroup() -> sig

    if (nrow(sig) == 0) next    
    
    sig$cluster = levels(group)[i]
    res = rbind(res,sig)
}

In [ ]:
write.csv(res,file = "./Results/Revision_2/DEG/Fibroblast_CT_edgeR_sig.csv")


In [ ]:
dt <- lapply(lapply(qlf, decideTests), summary)
dt.all <- do.call("cbind", dt)
dt.all

## Draw Bubble plot

In [ ]:
res = read.csv("./Results/Revision_2/DEG/Fibroblast_CT_edgeR_sig.csv",row.names = 1)
head(res)

In [ ]:
goi = c('LAMA2','TJP1','PDGFRA','DCN','COL1A2','COL3A1','SLC4A4','SLC47A1','MYRIP',"STXBP6",'STAT3','IL4R','IGFBP6','CLDN11')
    # 'Perivascular_fibroblast': ['LAMA2','TJP1','PDGFRA'],
    # 'Meningeal_fibroblast': ['SLC4A4','SLC47A1'],
    # 'Large_vessel_fibroblast': ['DCN','COL1A2','COL3A1'],
    # 'ChP_fibroblast': ['MYRIP',"STXBP6",'STAT3','IL4R'],
    # 'Base_barrier_cell': ['IGFBP6','CLDN11']
g_remove = c('KDR', 'IL4R', 'FLVCR2', 'CLDN11', 'CNTN5', 'LINGO2', 'CP', 'EYS')

res = res[!grepl("ENSG",res$genes),]
res = res[!grepl("LINC",res$genes),]
res %>%
  filter(logFC > 1, FDR < 0.05) %>%
  group_by(cluster) %>%
  arrange(FDR) %>%
  slice_head(n = 5) -> top

genes_oi = unique(c(top$genes,goi))
genes_oi = setdiff(genes_oi, g_remove)
length(genes_oi)

In [ ]:
## adjust the order
genes_oi = c('ABCA6','FBLN2','TNFRSF19','EDN3','KCNK17','PDGFRA','STAT3','NOX4','THBS2','RDH10','DCN','COL1A2','COL3A1','EBF2','SLC26A2','SLC47A1','SLC4A4','SLC9B2','IGFBP6','SLC13A3','MAPK4','SLC13A4','SLC22A6','MYRIP','SLC9A2','GCNT2','PTGDR','TJP1','STXBP6','DACH1','NPR3','TARID','RIPOR3','LAMA2')

In [ ]:
### Making sure get the corrected layer and matrix
DefaultAssay(obj_oi) = "RNA"

### Aggregation to pseudobulk 
mtx = AggregateExpression(
        obj_oi, 
        # features = gene_oi,
        group.by = c("Cell_type","individualID"),
        assays = 'RNA',
        slot = "counts"
    ) 
mtx <- as.matrix(mtx$RNA)
## Get the library size for each sample
lib_size <- Matrix::colSums(mtx)

### Get the logCPM from the matrix
cpm <- t(t(mtx) / lib_size) * 1e6
logCPM <- log2(cpm + 1)
logCPM = logCPM[genes_oi,]

### Calculate the averaged logCPM
# extract region (everything before first "_")
region <- sub("_.*$", "", colnames(logCPM))

# sanity check
table(region)

logCPM_region <- sapply(
  split(seq_len(ncol(logCPM)), region),
  function(i) {
    rowMeans(logCPM[, i, drop = FALSE])
  }
)
logCPM_region_z <- t(scale(t(logCPM_region)))

In [ ]:
mat = logCPM_region_z
all_genes = genes_oi
df_all = res
# ── 5. Build significance matrix ─────────────────────────────────────────────
sig_mat <- matrix(1,
                  nrow = length(all_genes),
                  ncol = length(unique(df_all$cluster)),
                  dimnames = list(all_genes, unique(df_all$cluster)))

for (j in seq_along(all_genes)) {
  for (i in seq_along(colnames(sig_mat))) {
    sub <- df_all[df_all$genes   == all_genes[j] &
                  df_all$cluster == colnames(sig_mat)[i], ]
    if (nrow(sub) > 0) sig_mat[j, i] <- sub$FDR
  }
}

# ── 6. Reorder columns by cluster number ─────────────────────────────────────
col_order <- order(as.numeric(str_split_fixed(colnames(mat), "_", 2)[, 1]))
mat     <- mat[, col_order]
sig_mat <- sig_mat[, col_order]

In [ ]:
# ── 8. Draw heatmap ──────────────────────────────────────────────────────────
ht <- Heatmap(
  mat,
  cluster_rows    = FALSE,
  cluster_columns = FALSE,
  col = colorRamp2(c(-1,0,1), c("#5387be","#F7F7F7","#b31f2c")),
  # right_annotation = row_ha,
  show_column_names = TRUE,
  show_row_names    = TRUE,
  column_title      = NULL,
  row_names_side    = "right",
  row_title_gp      = gpar(fontsize = 10),
  row_names_gp      = gpar(fontsize = 8),
  row_names_max_width = max_text_width(rownames(mat), gp = gpar(fontsize = 12)),
  cell_fun = function(j, i, x, y, w, h, fill) {
    if (sig_mat[i, j] < 0.05)
      grid.text("*", x, y, gp = gpar(fontsize = 25, col = "black"), vjust = "center")
  }
)
# pdf("./Results/Revision_2/DEG/Figure_4X_Fibroblast_DEG_heatmap.pdf",width = 4,height = 8)
# draw(ht)
# dev.off()
draw(ht)

In [ ]:
# Normalize the data
obj_oi <- NormalizeData(obj_oi)
# First check what levels you have
levels(factor(obj_oi$Cell_type))

# ── Option A: Custom manual order ────────────────────────────────────────────
cell_order <- rev(c('Fib_1','Fib_2','Fib_3','Fib_4','Fib_5','Fib_6'))   # replace with your actual cell type names
Idents(obj_oi) <- factor(obj_oi$Cell_type, levels = cell_order)

In [ ]:
options(repr.plot.height = 4,repr.plot.width = 10)
p1 <-  DotPlot(obj_oi,cols = viridis(3, option = "magma"),
             features = unique(genes_oi)) + RotatedAxis(180) +
                                    theme(axis.title = element_blank(),legend.position = "top")
p1

## Draw upset plot for the significant genes across all comparisons

In [ ]:
## get the significant genes
sig_genes_list = res %>%
  dplyr::filter(FDR < 0.05) %>%
  dplyr::filter(logFC > 0.5) %>%
  group_by(cluster) 

## check number of significant genes in each cluster
table(sig_genes_list$cluster)

In [ ]:
## Draw upset plot for the significant genes across clusters
library(UpSetR)
options(repr.plot.height = 6,repr.plot.width = 12)

## prepare the data for upset plot
sig_genes_list = res %>%
  dplyr::filter(FDR < 0.05) %>%
  dplyr::filter(logFC > 0.5) %>%
  # dplyr::filter(logFC < -0.5) %>%
  group_by(cluster) 
sig_genes_list = split(sig_genes_list$genes, sig_genes_list$cluster)
sig_genes_list = lapply(sig_genes_list, unique)
sig_genes_list = sig_genes_list[sapply(sig_genes_list, length) > 0]
# length(sig_genes_list)

## draw the upset plot and add title "Number of upregulated DEGs in each region cluster"
# pdf(file = "./Results/Revision_2/DEG/Fibroblast_sig_genes_upset_plot.pdf",width = 12,height = 6)
upset(
  fromList(sig_genes_list), 
  order.by = "freq", 
  nsets = length(sig_genes_list), 
  nintersects =NA,
#   main.bar.color = c("#FF0000","#0f0f0f","#03B8D8","#0f0f0f","#009E73", "#CC79A7","#009E73","#009E73","#0f0f0f","#0f0f0f", "#0f0f0f","#0f0f0f","#0f0f0f","#0f0f0f"),
#   sets.bar.color = c("#FF0000","#03B8D8", "#009E73", "#009E73", "#CC79A7"),
  text.scale = c(2, 2, 2, 2, 3, 2),
  point.size = 5, line.size = 1.6, mb.ratio = c(0.6, 0.4),
  att.pos = "bottom",
  mainbar.y.label = "Number of upregulated DEGs",
  sets.x.label = "Region Clusters"
  )
# dev.off()

### Perform GO enrichment analysis for the significant genes in each cluster


In [ ]:
suppressPackageStartupMessages({
library(clusterProfiler)
library('org.Hs.eg.db')
# library(ReactomePA)
})

In [ ]:
res = read.csv("./Results/Revision_2/DEG/Fibroblast_CT_edgeR_sig.csv",row.names = 1)
res_sig = res %>% dplyr::filter(FDR < 0.05) %>% dplyr::filter(logFC > 0.5)
table(res_sig$cluster)

In [ ]:
#########################################################################
#### find enriched GOBP for consistent signals in inhibitory neurons ####
#########################################################################
res$gene_id <- mapIds(
  # Replace with annotation package for the organism relevant to your data
  org.Hs.eg.db,
  # The vector of gene identifiers we want to map
  keys = res$genes,
  # Replace with the type of gene identifiers in your data
  keytype = "SYMBOL",
  # Replace with the type of gene identifiers you would like to map to
  column = "ENTREZID",
  multiVals = "first"
)

In [ ]:
### Run enrichment analysis for overlapped upregulated genes in C1, C2, and C3 clusters
sig_genes_list = res %>%
  dplyr::filter(FDR < 0.05) %>%
  dplyr::filter(logFC > 0.5) %>%
  group_by(cluster)
sig_genes_list = split(sig_genes_list$gene_id, sig_genes_list$cluster)
sig_genes_list = lapply(sig_genes_list, unique)
sig_genes_list = sig_genes_list[sapply(sig_genes_list, length) > 0]
length(sig_genes_list)

In [ ]:
# ## run Reactome pathway analysis for each cluster and combine the results
# era_result = data.frame()

# for (i in unique(res$cluster)){
#     cluster_genes <- res %>%
#         dplyr::filter(cluster == i, logFC > 0.5, FDR < 0.05) %>%
#         pull(gene_id) %>%
#         unique()
#     print(length(cluster_genes))

#     era = enrichPathway(gene=cluster_genes, pvalueCutoff = 0.05,readable = TRUE)
#     if (is.null(era)){
#             print(paste("No significant results for",i))
#         }else{
#             ## remove redundancy
#             era_df = era@result
#             era_df$cluster = i
#         }
#     era_result <- rbind(era_result, era_df)
# }

ego_result = data.frame()

for (i in unique(res$cluster)){
    cluster_genes <- res %>%
        dplyr::filter(cluster == i, logFC > 0.5, FDR < 0.05) %>%
        pull(gene_id) %>%
        unique()
    print(length(cluster_genes))

    ego <- enrichGO(gene = cluster_genes,
                OrgDb = org.Hs.eg.db,
                keyType = "ENTREZID",
                ont = "BP",
                pAdjustMethod = "BH",
                readable      = TRUE,
                pvalueCutoff = 0.05,
                qvalueCutoff = 0.05)
                
    if (is.null(ego)){
            print(paste("No significant results for",i))
        }else{
            ## remove redundancy
            ego_df = ego@result
            ego_df$cluster = i
        }
    ego_result <- rbind(ego_result, ego_df)
}

In [ ]:
unique(ego_result$cluster)

In [ ]:
# ##  Draw the heatmap top 5 significant pathways for each cluster and save the plot
# top_pathways <- era_result %>%
#   dplyr::filter(p.adjust < 0.05) %>%
#   group_by(cluster) %>%
#   arrange(p.adjust) %>%
#   slice_head(n = 5)
# top_pathways <- top_pathways %>%
#   ungroup() %>%
#   mutate(cluster = factor(cluster, levels = c("Fib-1","Fib-2","Fib-3","Fib-4","Fib-5",'Fib-6')))

# options(repr.plot.width = 13,repr.plot.height = 9)
# p2 <- ggplot(top_pathways, aes(x = cluster, y = Description, fill = -log10(p.adjust))) +
#   geom_tile() +
#   scale_fill_viridis(begin = 0,end = 1,option = "viridis") +
#   theme_classic() +
#   theme(axis.text.x = element_text(angle = 45, hjust = 1,size = 18),
#         axis.text.y = element_text(size = 18))
# p2

##  Draw the heatmap top 5 significant pathways for each cluster and save the plot
top_pathways <- ego_result %>%
  dplyr::filter(p.adjust < 0.05) %>%
  group_by(cluster) %>%
  arrange(p.adjust) %>%
  slice_head(n = 5)

top_pathways <- top_pathways %>%
  ungroup() %>%
  mutate(
    cluster = factor(cluster, levels = c('Fib-1','Fib-2','Fib-3','Fib-4','Fib-5','Fib-6')),
    Description_wrapped = str_wrap(Description, width = 70)
  ) %>%
  # sort the data frame by cluster then FDR before creating the factor
  arrange(cluster, p.adjust) %>%
  mutate(
    Description_wrapped = factor(Description_wrapped, 
                                 levels = rev(unique(Description_wrapped)))
  )

options(repr.plot.width = 5, repr.plot.height = 4)

p3 <- ggplot(top_pathways, aes(x = cluster, y = Description_wrapped, 
                                fill = -log10(p.adjust))) +
  geom_tile(color = "white", linewidth = 0.4) +
  scale_fill_gradientn(
    colours = c("#F5F5F5", "#26A69A", "#00695C"),   # white → coral → dark red
    name    = expression(-log[10](p.adjust))
  ) +
  scale_y_discrete(labels = function(x) x) +         # keeps \n line breaks intact
  labs(x = NULL, y = NULL) +
  theme_classic() +
  theme(
    axis.text.x  = element_text(angle = 45, hjust = 1, size = 10),
    axis.text.y  = element_text(size = 8, lineheight = 0.85),  # tighten wrapped lines
    legend.title = element_text(size = 9),
    legend.text  = element_text(size = 6),
    panel.border = element_rect(color = "grey80", fill = NA, linewidth = 0.5)
  )
pdf(file = "./Results/Revision_2/DEG/Figure_4X_Fibroblasts_cell_type_top_pathways_go_heatmap.pdf",width = 6.5,height = 6)
p3
dev.off()

In [ ]:
## get meta data to work on
meta <- obj_oi@meta.data

## get the count table
counts <- meta %>%
  group_by(Cell_type, region_layer) %>%
  summarise(freq = n(), .groups = "drop")
colnames(counts) <- c("Cell_type","brain_region","freq")

# ## reorder the brain region based on the number code
# counts$order <- as.numeric(str_split_fixed(counts$brain_region,pattern = "_",n = 2)[,1])
# counts <- counts[order(counts$Cell_type),]
# counts <- counts[order(counts$order),]

# ## making sure corrected order
# counts$brain_region_plot = str_split_fixed(counts$brain_region,pattern = "_",n = 2)[,2]
# counts$brain_region_plot<- factor(counts$brain_region_plot, levels=unique(counts$brain_region_plot))


counts$Cell_type <- factor(counts$Cell_type)
col1=c(
  "Fib_1" = "#00695C",
  "Fib_2" = "#00796B",
  "Fib_3" = "#009688",
  "Fib_4" = "#26A69A",
  "Fib_5" = "#4DB6AC",
  "Fib_6" = "#80CBC4"
)

counts$color <- col1[counts$Cell_type]

In [ ]:
p4 <- ggplot(counts, aes(fill=Cell_type, y=freq, x=region_layer)) + 
        geom_bar(position="fill", stat="identity") +
        theme_minimal() +
        RotatedAxis() + 
        theme(axis.text.x = element_text(angle = 90, vjust = 0.5, hjust = 1, size = 15),
              axis.text.y = element_text(size = 15),
              axis.title.y = element_text(size = 15)) +
        scale_fill_manual(values=col1) +
        xlab("") +
        ylab("Frequency")
options(repr.plot.width = 12, repr.plot.height = 6.5, repr.plot.res = 100) # set the resolution
# p3
# p4
p4
# ggsave(filename = "./Results/Revision_2/Figures/Figure_1X_Fibroblast_type_region_proportion.pdf", 
# patchwork::wrap_plots(p4, ncol = 1),
# scale = 1, width = 12, height = 7.5)

## Chi-sq to check which brain region contain more of which fibroblast

In [ ]:
meta = obj_oi@meta.data
table(meta$Cell_type)

In [ ]:
### try chisq contigency table
meta <- obj_oi@meta.data
# meta$brain_region <- paste(meta$regionID,str_to_title(str_trim(meta$region_name)),sep = "_")

obs_mat = as.matrix(table(meta$region_abb,meta$Cell_type))

In [ ]:
# run chisq test
chisq_res <- chisq.test(obs_mat)
exp_mat = chisq_res$expected

# get the log1p(mat)
res = (obs_mat/exp_mat)+1
res = log(res)
# res = res[order(as.numeric(str_split_fixed(rownames(res),pattern = "_",n = 2)[,1])),]

In [ ]:
res = t(res)
res

In [ ]:
## Provide the color code
meta <- obj_oi@meta.data
df <- unique(meta[,c("region_abb","region_layer")])
# df$region_name = str_to_title(str_trim(df$region_name))
df = as.data.frame(df)

df$region_color <- NA
df[df$region_layer == "Cortex",]$region_color = "#009E73"
df[df$region_layer == "Brainstem",]$region_color = "#D55E00"
df[df$region_layer == "Cerebellum",]$region_color = "#046299"
df[df$region_layer == "Limbic",]$region_color = "#03B8D8"
df[df$region_layer == "Watershed",]$region_color = "#E69F00"
df[df$region_layer == "White Matter Tracts",]$region_color = "#CC79A7"
df[df$region_layer == "Olfactory",]$region_color = "#999999"
df[df$region_layer == "Barrier",]$region_color = "#9C0AE0"
# df[df$region_layer == "Subcortical",]$region_color = "#915330"
df[df$region_layer == "Major Vessel",]$region_color = "#FF0000"

# ## Assign region color
region_color <- df$region_color
names(region_color) <-df$region_abb

In [ ]:
# annotation for heatmap
hl <- HeatmapAnnotation(
  region = colnames(res),
#   avg_density = avg_vec,
  col = list(
    region = region_color
  ),
  annotation_name_side = "right"
)

In [ ]:
## plotting
library(ComplexHeatmap)
library(colorRamp2)

#options(repr.plot.width = 10, repr.plot.height = 12, repr.plot.res = 100) # set the resolution
p = Heatmap(res,col = colorRamp2(c(0,1.5,3),c("grey95","red","red4")),cluster_rows = F,cluster_columns = T,top_annotation = hl)

pdf("./Results/Revision_2/Figures/Figure_4X_Fibroblast_Cell_type_enrichment_cluster.pdf",width = 13,height = 3.5)
p
dev.off()

In [ ]:
p

In [ ]:
sessionInfo()